# Space risk model — answers to questions 1–10
The notebook reads `space_risk_model.csv` and writes `submission_space_risk_model.csv`.

In [1]:
import itertools, time
import numpy as np
import pandas as pd

DATA = "space_risk_model.csv"
df = pd.read_csv(DATA)
df = df.sort_values(["satellite_id", "orbit_number"], kind="stable").reset_index(drop=True)
MODELS = [f"model_{i}_pred" for i in range(1, 11)]
answers = {}
print(df.shape, "failure rate:", round(df.failure_in_12h.mean(), 4))

(180300, 26) failure rate: 0.0204


## Q1. Satellite with the highest mean solar panel temperature per orbit type

In [2]:
sat = df.groupby(["orbit_type", "satellite_id"])["solar_panel_temp_mean"].mean().reset_index()
q1 = [int(sat.loc[sat[sat.orbit_type == ot].solar_panel_temp_mean.idxmax(), "satellite_id"]) for ot in (0, 1, 2)]
answers[1] = [str(x) for x in q1]
print(answers[1])

['15', '3', '1']


## Q2. Top-5 models by PR-AUC (Average Precision)
Rare positive class (~2%) → PR-AUC is the relevant threshold-independent ranking metric. Ties keep the table order (stable sort), no merging of equal predictions.

In [3]:
y = df.failure_in_12h.to_numpy()

def average_precision_full(y, score):
    order = np.argsort(-score, kind="stable")   # descending, ties keep table order
    ys = y[order]
    tp = np.cumsum(ys)
    precision = tp / np.arange(1, len(ys) + 1)
    return precision[ys == 1].sum() / ys.sum()

ap = {i: average_precision_full(y, df[m].to_numpy()) for i, m in enumerate(MODELS, 1)}
for k, v in ap.items():
    print(k, round(v, 4))
top = sorted(ap, key=lambda k: (-ap[k], k))[:5]
answers[2] = [",".join(map(str, top)), ",".join(f"{ap[k]:.2f}" for k in top)]
print(answers[2])

1 0.0946
2 0.0384
3 0.0313
4 0.1621
5 0.2743
6 0.0512
7 0.3143
8 0.07
9 0.4804
10 0.92
['10,9,7,5,4', '0.92,0.48,0.31,0.27,0.16']


## Q3. Heating rate 3 orbits before a failure vs. orbits without failure

In [4]:
g = df.groupby("satellite_id", sort=False)
hr = df.solar_panel_temp_mean - g.solar_panel_temp_mean.shift(3)      # undefined for first 3 orbits
hr_abs = hr.abs()
hr_abs_k3 = hr_abs.groupby(df.satellite_id, sort=False).shift(3)      # |heating_rate| at orbit k-3
no_fail = hr_abs[(df.failure_in_12h == 0) & hr_abs.notna()].mean()
pre_fail = hr_abs_k3[(df.failure_in_12h == 1) & hr_abs_k3.notna()].mean()
print(pre_fail, no_fail, pre_fail / no_fail)
answers[3] = [str(int(round(pre_fail / no_fail)))]
print(answers[3])

3.683184676381616 0.4988558076587179 7.383265103533469
['7']


## Q4. Minimal FPR at Recall = 1

In [5]:
best4 = None
for i, m in enumerate(MODELS, 1):
    p = df[m].to_numpy()
    t = p[y == 1].min()
    pred = p >= t
    fp = (pred & (y == 0)).sum(); tn = (~pred & (y == 0)).sum()
    fpr = fp / (fp + tn)
    print(i, round(t, 6), round(fpr, 4))
    if best4 is None or fpr < best4[1]:
        best4 = (i, fpr)
answers[4] = [str(best4[0]), f"{best4[1]:.2f}"]
print(answers[4])

1 0.01 1.0
2 0.01 1.0
3 0.01 1.0
4 0.01 1.0
5 0.01 1.0
6 0.01 1.0
7 0.01 1.0
8 0.01 1.0
9 0.01 1.0
10 0.333592 0.2887
['10', '0.29']


## Q5. Lagged cross-correlation solar panels vs. batteries

In [6]:
mean_r = {}
for lag in range(-5, 6):
    rs = []
    for _, s in df.groupby("satellite_id", sort=False):
        a = s.solar_panel_temp_mean.to_numpy(); b = s.battery_temp_mean.to_numpy()
        if lag > 0:   x, z = a[:-lag], b[lag:]
        elif lag < 0: x, z = a[-lag:], b[:lag]
        else:         x, z = a, b
        rs.append(np.corrcoef(x, z)[0, 1])
    mean_r[lag] = np.mean(rs)
    print(lag, round(mean_r[lag], 4))
best_lag = max(mean_r, key=mean_r.get)
answers[5] = [str(best_lag), f"{mean_r[best_lag]:.2f}"]
print(answers[5])

-5 -0.1383
-4 -0.0668
-3 0.0157
-2 0.1139
-1 0.2955
0 0.5593
1 0.7798
2 0.4383
3 0.2141
4 0.0646
5 -0.0177
['1', '0.78']


## Q6. Partial correlations

In [7]:
def partial_corr(x, y_, z):
    rxy = np.corrcoef(x, y_)[0, 1]; rxz = np.corrcoef(x, z)[0, 1]; ryz = np.corrcoef(y_, z)[0, 1]
    return (rxy - rxz * ryz) / np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2))

p1 = partial_corr(df.solar_panel_temp_mean, df.battery_current_mean, df.battery_voltage_mean)
p2 = partial_corr(df.battery_current_mean, df.battery_voltage_mean, df.solar_panel_temp_mean)
print(p1, p2)
answers[6] = [f"{max(abs(p1), abs(p2)):.2f}"]
print(answers[6])

0.11007541670504788 -0.24212030239677143
['0.24']


## Q7. Maximum F1 of the linear risk index (weights grid 0.01, all distinct thresholds, R > t)

In [8]:
a7 = df.battery_temp_mean.to_numpy() / 40
b7 = df.reaction_wheel_current_mean.to_numpy() / 2.0
c7 = df.attitude_error_mean.to_numpy() / 1.5
npos = y.sum()

def best_f1(score):
    order = np.argsort(-score, kind="stable")
    s, ys = score[order], y[order]
    tp = np.cumsum(ys)
    k = np.arange(1, len(ys) + 1)
    cut = np.r_[s[1:] != s[:-1], True]       # top-k dangerous is a valid split only at value changes
    return (2 * tp / (k + npos))[cut].max()

best7 = (-1, None)
for i in range(101):
    for j in range(101 - i):
        w = (i / 100, j / 100, (100 - i - j) / 100)
        f = best_f1(w[0] * a7 + w[1] * b7 + w[2] * c7)
        if f > best7[0]:
            best7 = (f, w)
print(best7)
answers[7] = [f"{best7[0]:.3f}"]
print(answers[7])

(np.float64(0.9446145282493738), (0.18, 0.59, 0.23))
['0.945']


## Q8. Share of anomalous orbits vs. number of failures

In [9]:
rows = []
for sid, s in df.groupby("satellite_id", sort=False):
    x = s.reaction_wheel_current_std.to_numpy()
    thr = x.mean() + 3 * x.std(ddof=0)
    rows.append(((x > thr).mean(), s.failure_in_12h.sum()))
rows = np.array(rows, dtype=float)
r8 = np.corrcoef(rows[:, 0], rows[:, 1])[0, 1]
answers[8] = [f"{r8:.3f}"]
print(r8, answers[8])

0.693383832031676 ['0.693']


## Q9. Residual std ratio of the linear regression

In [10]:
X = np.column_stack([np.ones(len(df)), df.attitude_error_mean, df.reaction_wheel_current_mean])
t9 = df.battery_temp_mean.to_numpy()
beta, *_ = np.linalg.lstsq(X, t9, rcond=None)
res = t9 - X @ beta
ratio9 = res[y == 1].std(ddof=1) / res[y == 0].std(ddof=1)
answers[9] = [f"{ratio9:.2f}"]
print(beta, ratio9, answers[9])

[30.46459883  0.14628299  0.21655266] 0.9802543278100321 ['0.98']


## Q10. Optimal 3-model ensemble under the loss model and constraints
Weights on a grid with step `STEP` (all weights > 0); for every combination all distinct thresholds are tried (P > t).

In [11]:
STEP = 0.05
C_FN, C_FP = 20_000_000, 100_000
grp = df.orbit_type.to_numpy()
preds = {i: df[m].to_numpy() for i, m in enumerate(MODELS, 1)}
N = len(y)
gmask = [grp == k for k in (0, 1, 2)]
gN = [m.sum() for m in gmask]
gP = [y[m].sum() for m in gmask]
u = int(round(1 / STEP))
weights = [(i / u, j / u, (u - i - j) / u) for i in range(1, u) for j in range(1, u - i)]

best10 = None   # (loss, t, triple, weights)
t0 = time.time()
for triple in itertools.combinations(range(1, 11), 3):
    pa, pb, pc = (preds[k] for k in triple)
    for w in weights:
        p = w[0] * pa + w[1] * pb + w[2] * pc
        order = np.argsort(-p, kind="stable")
        ps, ys = p[order], y[order]
        k = np.arange(1, N)                       # top-k orbits are dangerous, t = ps[k]
        valid = ps[1:] < ps[:-1]
        tp = np.cumsum(ys)[:-1]
        fn = npos - tp
        fp = k - tp
        ok = valid & (fn <= 0.10 * npos) & (k <= 0.25 * N)
        for gi in range(3):
            gm = gmask[gi][order]
            gk = np.cumsum(gm)[:-1]
            gtp = np.cumsum(gm & (ys == 1))[:-1]
            ok &= ((gP[gi] - gtp) <= 0.15 * gP[gi]) & (gk <= 0.30 * gN[gi])
        if not ok.any():
            continue
        loss = np.where(ok, C_FN * fn + C_FP * fp, np.iinfo(np.int64).max)
        lmin = loss.min()
        t = ps[np.nonzero(loss == lmin)[0] + 1].min()
        cand = (int(lmin), t, triple, w)
        if best10 is None or cand[:3] < best10[:3]:
            best10 = cand
print(best10, f"{time.time() - t0:.0f}s")
answers[10] = [",".join(map(str, best10[2])), f"{best10[1]:.2f}", str(best10[0])]
print(answers[10])

(181900000, np.float64(0.3726815758697751), (7, 9, 10), (0.45, 0.2, 0.35)) 616s
['7,9,10', '0.37', '181900000']


## Submission file

In [12]:
import csv
with open("submission_space_risk_model.csv", "w", newline="") as fh:
    wr = csv.writer(fh)
    wr.writerow(["question_id", "answer_1", "answer_2", "answer_3"])
    for q in range(1, 11):
        a = answers[q]
        wr.writerow([q] + a + [""] * (3 - len(a)))
print(open("submission_space_risk_model.csv").read())

question_id,answer_1,answer_2,answer_3
1,15,3,1
2,"10,9,7,5,4","0.92,0.48,0.31,0.27,0.16",
3,7,,
4,10,0.29,
5,1,0.78,
6,0.24,,
7,0.945,,
8,0.693,,
9,0.98,,
10,"7,9,10",0.37,181900000

